# Trabajo práctico - Asignatura 7

**Caso de uso:** clasificación del `Status` de activos del inventario WMS.

Este notebook reproduce el flujo principal del trabajo: lectura del XLSX, depuración, ingeniería de variables, entrenamiento, ajuste, evaluación y segmentación.

## Resumen ejecutivo

- Registros brutos: **999**
- Registros válidos para modelado: **797**
- Registros filtrados por inconsistencias o ausencia de etiqueta: **202**
- Mejor modelo: **Random forest**
- Accuracy test: **0.9437**
- F1 macro test: **0.9373**

In [ ]:

import zipfile, xml.etree.ElementTree as ET, re, collections, statistics
import numpy as np

path = "wms_dataset_clean (2).xlsx"
ns = {'a':'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}

zf = zipfile.ZipFile(path)
root = ET.fromstring(zf.read('xl/sharedStrings.xml'))
shared = [''.join(t.text or '' for t in si.iter('{http://schemas.openxmlformats.org/spreadsheetml/2006/main}t'))
          for si in root.findall('a:si', ns)]

def col_to_idx(col):
    n = 0
    for ch in col:
        if ch.isalpha():
            n = n * 26 + (ord(ch.upper()) - 64)
    return n - 1

sheet = ET.fromstring(zf.read('xl/worksheets/sheet1.xml'))
rows = []
for row in sheet.findall('.//a:sheetData/a:row', ns):
    vals = {}
    for c in row.findall('a:c', ns):
        m = re.match(r'([A-Z]+)(\d+)', c.attrib.get('r',''))
        if not m:
            continue
        col = col_to_idx(m.group(1))
        t = c.attrib.get('t')
        v = c.find('a:v', ns)
        isel = c.find('a:is', ns)
        if t == 's' and v is not None:
            value = shared[int(v.text)]
        elif t == 'inlineStr' and isel is not None:
            value = ''.join(tn.text or '' for tn in isel.iter('{http://schemas.openxmlformats.org/spreadsheetml/2006/main}t'))
        elif v is not None:
            value = v.text
        else:
            value = None
        vals[col] = value
    rows.append([vals.get(i) for i in range(16)])

headers = rows[0][:16]
data = [r[:16] for r in rows[1:]]
print("Registros brutos:", len(data))
print("Columnas:", headers)


## Limpieza y definición del problema

Se plantea un problema de **clasificación multiclase** sobre la variable `Status` con tres categorías: `In Use`, `In Storage` y `Needs Repair`.

Se eliminan registros con categorías incoherentes o sin `Status`, y se derivan variables temporales desde `Purchase Date`.

In [ ]:

valid_categories = {'Hardware','Tools','Consumable','Supplies','Books/Documents'}
valid_status = {'In Use','In Storage','Needs Repair'}

clean = [r for r in data if len(r) >= 16 and r[4] in valid_categories and r[6] in valid_status]

records = []
for r in clean:
    m = re.match(r'(\d{2})-(\d{2})-(\d{4})', r[10] or '')
    month = year = None
    if m:
        _, month, year = map(int, m.groups())
    records.append({
        'description': r[2],
        'quantity': float(r[3]) if r[3] else None,
        'category': r[4],
        'department': r[5],
        'condition': r[7],
        'campus': r[8],
        'location': r[9],
        'purchase_year': year,
        'purchase_month': month,
        'has_po': 1 if r[12] else 0,
        'supplier': r[13] if r[13] else 'Unknown',
        'status': r[6]
    })

print("Registros válidos:", len(records))
print("Distribución de status:", collections.Counter(rec['status'] for rec in records))


## Preparación de variables

Se usa un `ColumnTransformer` con tres bloques:

- **Texto** (`Description`) con TF-IDF.
- **Numéricas** (`quantity`, `purchase_year`, `purchase_month`, `has_po`) con imputación por mediana y `StandardScaler`.
- **Categóricas** (`category`, `department`, `condition`, `campus`, `location`, `supplier`) con imputación por moda y one-hot encoding.

In [ ]:

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

feature_names = ['description','quantity','category','department','condition','campus','location','purchase_year','purchase_month','has_po','supplier']
X = np.array([[rec[f] for f in feature_names] for rec in records], dtype=object)
y = np.array([rec['status'] for rec in records])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocess = ColumnTransformer([
    ('text', TfidfVectorizer(max_features=120, ngram_range=(1,2), min_df=2), 0),
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), [1,7,8,9]),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), [2,3,4,5,6,10])
])

baseline = Pipeline([
    ('preprocess', preprocess),
    ('model', DummyClassifier(strategy='most_frequent'))
]).fit(X_train, y_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

log_pipe = Pipeline([
    ('preprocess', preprocess),
    ('model', LogisticRegression(max_iter=3000, class_weight='balanced'))
])

rf_pipe = Pipeline([
    ('preprocess', preprocess),
    ('model', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

gs_log = GridSearchCV(
    log_pipe,
    {'model__C':[0.3,1,3], 'preprocess__text__max_features':[80,120]},
    cv=cv, scoring='f1_macro', n_jobs=1
).fit(X_train, y_train)

gs_rf = GridSearchCV(
    rf_pipe,
    {'model__n_estimators':[200], 'model__max_depth':[None,12,20], 'model__min_samples_leaf':[1,2,4], 'preprocess__text__max_features':[80,120]},
    cv=cv, scoring='f1_macro', n_jobs=1
).fit(X_train, y_train)

pred_base = baseline.predict(X_test)
pred_log = gs_log.best_estimator_.predict(X_test)
pred_rf = gs_rf.best_estimator_.predict(X_test)

print("Baseline:", accuracy_score(y_test,pred_base), f1_score(y_test,pred_base,average='macro'))
print("Logística:", accuracy_score(y_test,pred_log), f1_score(y_test,pred_log,average='macro'))
print("Random forest:", accuracy_score(y_test,pred_rf), f1_score(y_test,pred_rf,average='macro'))
print("Mejor configuración logística:", gs_log.best_params_)
print("Mejor configuración RF:", gs_rf.best_params_)


## Resultados obtenidos

- **Baseline**: accuracy 0.5188, F1 macro 0.2277
- **Regresión logística**: accuracy 0.9375, F1 macro 0.9330
- **Random forest**: accuracy 0.9437, F1 macro 0.9373

El modelo final seleccionado es **Random forest**, por ofrecer el mejor equilibrio entre exactitud y F1 macro.

## Matriz de confusión del modelo final

Clases: In Storage, In Use, Needs Repair

```text
[[75, 8, 0], [0, 70, 0], [0, 1, 6]]
```

In [ ]:

from sklearn.feature_extraction import DictVectorizer
from sklearn.cluster import KMeans

loc_counts = collections.Counter(rec['location'] for rec in records)
top_locs = {loc for loc,_ in loc_counts.most_common(8)}

cluster_dicts = []
for rec in records:
    cluster_dicts.append({
        'quantity': rec['quantity'],
        'category': rec['category'],
        'department': rec['department'],
        'condition': rec['condition'],
        'campus': rec['campus'],
        'location_group': rec['location'] if rec['location'] in top_locs else 'Other',
        'purchase_year': rec['purchase_year'],
        'has_po': rec['has_po']
    })

vec = DictVectorizer(sparse=False)
Xc = vec.fit_transform(cluster_dicts)
Xc = StandardScaler().fit_transform(Xc)

km = KMeans(n_clusters=3, random_state=42, n_init=10, max_iter=100)
labels = km.fit_predict(Xc)

for cl in sorted(set(labels)):
    subset = [records[i] for i,l in enumerate(labels) if l == cl]
    print(f"Cluster {cl} - tamaño {len(subset)}")
    print("Status:", collections.Counter(rec['status'] for rec in subset).most_common(3))
    print("Departamento:", collections.Counter(rec['department'] for rec in subset).most_common(3))
    print("Ubicación:", collections.Counter(rec['location'] for rec in subset).most_common(3))
    print("-"*60)


## Conclusión

El trabajo demuestra que el dataset WMS permite construir un modelo predictivo útil para anticipar el estado operativo de los activos. El enfoque puede integrarse en un sistema de soporte a decisiones para inventario, mantenimiento y despliegue de equipamiento.